# NMF Topic Model Evaluation — Coherence, Interpretability, Diversity & Best *k*

Systematic evaluation of NMF topic models on the ResearchLanka publication corpus.
Uses `src.modeling.nmf_topic_modeling` (same logic as `scripts/modeling/run_nmf_topic_modeling.py`).

## Metrics

| Metric | Direction | Meaning |
|--------|-----------|--------|
| **Coherence (c_v)** | higher ↑ | Top topic words co-occur in real documents (gensim) |
| **Topic diversity (TD)** | higher ↑ | Share of unique words across topics' top-N lists |
| **Pairwise redundancy** | lower ↓ | Mean Jaccard overlap between topic word sets |
| **Reconstruction error** | lower ↓ | NMF fit quality on the TF-IDF matrix |

## Workflow

1. Load corpus → build TF-IDF → sweep *k*
2. Plot all metrics vs *k*
3. Rank *k* with a composite score (coherence + diversity − redundancy)
4. Inspect top keywords at candidate *k* values
5. Export annotation template for human interpretability scoring
6. Fit final model and save artifacts

**Outputs:** `./outputs/nmf_evaluation/`

## 1. Setup

Requires `gensim` for coherence scoring (not in the base package):

```bash
pip install gensim matplotlib seaborn
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

BACKEND_ROOT = Path("..").resolve()
sys.path.insert(0, str(BACKEND_ROOT))

from src.modeling.nmf_topic_modeling import (
    TEXT_COLUMNS,
    build_tfidf,
    build_annotation_template,
    combined_text,
    compute_pairwise_redundancy,
    compute_topic_diversity,
    evaluate_k_range,
    find_year_column,
    get_topic_keywords,
    name_topics,
    pick_best_k,
    run_final_pipeline,
)
from src.preprocessing.text_cleaning import cleaning_report

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

## 2. Configuration

In [ ]:
DATA_PATH = BACKEND_ROOT / "data/processed/common/common_publications_final_with_linearsvm.csv"
# Fallback if the SVM-enriched file is not present:
if not DATA_PATH.exists():
    DATA_PATH = BACKEND_ROOT / "data/processed/common/common_publications_final.csv"

OUTPUT_DIR = Path("./outputs/nmf_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_RANGE = [5, 10, 15, 20, 25, 30]  # full sweep range
N_WORDS = 15                        # top words kept per topic
NAMING_WORDS = 3                      # words in auto-generated topic labels
RANDOM_STATE = 42
CLEAN_TEXT = True                     # strip Tamil/Sinhala + boilerplate (see text_cleaning.py)

# Set to an integer for a fast smoke test; None = full corpus
SAMPLE_N = None

print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR.resolve()}")

## 3. Load corpus & text cleaning audit

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
if SAMPLE_N:
    df = df.sample(SAMPLE_N, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Corpus shape: {df.shape}")

raw_texts = combined_text(df, TEXT_COLUMNS, clean=False)
report = cleaning_report(raw_texts)
pd.DataFrame([report])

In [ ]:
texts = combined_text(df, TEXT_COLUMNS, clean=CLEAN_TEXT)
has_text = texts.str.len() > 0
print(f"Rows with usable text: {has_text.sum():,} / {len(df):,}")

vectorizer, X = build_tfidf(texts[has_text])
feature_names = vectorizer.get_feature_names_out()
print(f"TF-IDF matrix: {X.shape[0]:,} docs × {X.shape[1]:,} features")

## 4. Sweep *k* — coherence, diversity, redundancy, reconstruction error

Passing the fitted `vectorizer` ensures coherence tokenization matches the n-gram
vocabulary (`ngram_range=(1, 3)`) used to build topics.

In [ ]:
summary_df, results = evaluate_k_range(
    X,
    feature_names,
    texts[has_text],
    K_RANGE,
    vectorizer=vectorizer,
    n_words=N_WORDS,
    random_state=RANDOM_STATE,
)
summary_df

In [ ]:
summary_df.to_csv(OUTPUT_DIR / "k_sweep_metrics.csv", index=False)
summary_df.describe().round(4)

## 5. Visualize metrics vs *k*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
metrics = [
    ("coherence_cv", "Coherence (c_v)", "higher is better", "tab:blue"),
    ("diversity", "Topic diversity (TD)", "higher is better", "tab:green"),
    ("redundancy", "Pairwise redundancy (Jaccard)", "lower is better", "tab:red"),
    ("reconstruction_error", "Reconstruction error", "lower is better", "tab:purple"),
]

for ax, (col, title, hint, color) in zip(axes.ravel(), metrics):
    ax.plot(summary_df["k"], summary_df[col], marker="o", color=color, linewidth=2)
    ax.set_xlabel("Number of topics (k)")
    ax.set_ylabel(title)
    ax.set_title(f"{title}\n({hint})")
    ax.set_xticks(summary_df["k"])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "k_sweep_metrics.png", bbox_inches="tight")
plt.show()

## 6. Composite score — choose best *k*

Coherence alone can favour redundant topics. We combine three interpretability-related
signals (each min–max normalized across the sweep):

$$\text{composite} = 0.50 \cdot \text{coherence} + 0.30 \cdot \text{diversity} + 0.20 \cdot (1 - \text{redundancy})$$

Reconstruction error is reported but not in the composite — it keeps decreasing with larger *k*
and does not reflect topic quality on its own.

In [ ]:
def minmax(series: pd.Series) -> pd.Series:
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(1.0, index=series.index)
    return (series - lo) / (hi - lo)


def rank_k(summary: pd.DataFrame) -> pd.DataFrame:
    ranked = summary.copy()
    ranked["coherence_norm"] = minmax(ranked["coherence_cv"])
    ranked["diversity_norm"] = minmax(ranked["diversity"])
    ranked["redundancy_inv_norm"] = 1 - minmax(ranked["redundancy"])
    ranked["composite_score"] = (
        0.50 * ranked["coherence_norm"]
        + 0.30 * ranked["diversity_norm"]
        + 0.20 * ranked["redundancy_inv_norm"]
    )
    ranked = ranked.sort_values("composite_score", ascending=False).reset_index(drop=True)
    return ranked


ranked_df = rank_k(summary_df)
best_k_coherence = pick_best_k(summary_df)
best_k_composite = int(ranked_df.loc[0, "k"])

print(f"Best k by coherence alone:     {best_k_coherence}")
print(f"Best k by composite score:     {best_k_composite}")
print()
ranked_df[[
    "k", "coherence_cv", "diversity", "redundancy",
    "coherence_norm", "diversity_norm", "redundancy_inv_norm", "composite_score",
]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = ranked_df.sort_values("k")["k"]
ax.bar(
    ranked_df.sort_values("k")["k"].astype(str),
    ranked_df.sort_values("k")["composite_score"],
    color="steelblue",
    alpha=0.85,
)
ax.axvline(
    str(best_k_composite),
    color="crimson",
    linestyle="--",
    label=f"Best composite k={best_k_composite}",
)
ax.set_xlabel("k")
ax.set_ylabel("Composite score")
ax.set_title("Composite topic quality score by k")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "k_composite_scores.png", bbox_inches="tight")
plt.show()

## 7. Inspect topic keywords at top candidate *k* values

Review the top 3 *k* values by composite score before locking in a final choice.

In [ ]:
CANDIDATE_KS = ranked_df.head(3)["k"].astype(int).tolist()
print("Candidate k values:", CANDIDATE_KS)

keyword_rows = []
for k in CANDIDATE_KS:
    topic_words = results[k]["topic_words"]
    topic_names = name_topics(topic_words, n=NAMING_WORDS)
    for topic_id, (name, words) in enumerate(zip(topic_names, topic_words), start=1):
        keyword_rows.append({
            "k": k,
            "topic_id": topic_id,
            "auto_name": name,
            "top_words": ", ".join(words[:10]),
        })

keywords_compare_df = pd.DataFrame(keyword_rows)
keywords_compare_df.to_csv(OUTPUT_DIR / "candidate_k_keywords.csv", index=False)

for k in CANDIDATE_KS:
    print(f"\n{'=' * 60}\nk = {k}\n{'=' * 60}")
    subset = keywords_compare_df[keywords_compare_df["k"] == k]
    for _, row in subset.iterrows():
        print(f"  T{row['topic_id']:02d} [{row['auto_name']}]: {row['top_words']}")

## 8. Interpretability — topic word overlap heatmap

High off-diagonal Jaccard similarity means two topics share vocabulary (redundant / hard to interpret).

In [ ]:
def topic_jaccard_matrix(topic_words: list[list[str]], top_n: int = 15) -> np.ndarray:
    sets = [set(w[:top_n]) for w in topic_words]
    n = len(sets)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            union = sets[i] | sets[j]
            mat[i, j] = len(sets[i] & sets[j]) / len(union) if union else 0.0
    return mat


INSPECT_K = best_k_composite  # change manually after reading keywords above
inspect_words = results[INSPECT_K]["topic_words"]
inspect_names = name_topics(inspect_words, n=NAMING_WORDS)

jaccard = topic_jaccard_matrix(inspect_words, top_n=N_WORDS)
labels = [f"T{i+1}: {n[:30]}" for i, n in enumerate(inspect_names)]

fig, ax = plt.subplots(figsize=(max(8, INSPECT_K * 0.45), max(6, INSPECT_K * 0.4)))
sns.heatmap(
    jaccard,
    xticklabels=labels,
    yticklabels=labels,
    annot=INSPECT_K <= 15,
    fmt=".2f",
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    ax=ax,
)
ax.set_title(f"Topic word Jaccard similarity (k={INSPECT_K}, top {N_WORDS} words)")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"topic_overlap_k{INSPECT_K}.png", bbox_inches="tight")
plt.show()

print(f"Diversity (TD): {compute_topic_diversity(inspect_words, top_n=N_WORDS):.3f}")
print(f"Mean pairwise redundancy: {compute_pairwise_redundancy(inspect_words, top_n=N_WORDS):.3f}")

## 9. Human interpretability template

Automated diversity/redundancy are proxies. Fill in `manual_name` and
`interpretability_score_1to5` (1 = nonsense, 5 = clearly named research area) for each topic.

In [ ]:
annotation_df = build_annotation_template(inspect_words, inspect_names, n_words=10)
annotation_df.to_csv(OUTPUT_DIR / f"annotation_template_k{INSPECT_K}.csv", index=False)
annotation_df

## 10. Final *k* selection

Default: composite-score winner. Override `FINAL_K` after reviewing keywords and the annotation template.

In [ ]:
FINAL_K = best_k_composite  # override manually if needed, e.g. FINAL_K = 20

final_metrics = summary_df.loc[summary_df["k"] == FINAL_K].iloc[0]
print(f"Selected k = {FINAL_K}")
print(f"  coherence_cv:         {final_metrics['coherence_cv']:.4f}")
print(f"  diversity:            {final_metrics['diversity']:.3f}")
print(f"  redundancy:           {final_metrics['redundancy']:.3f}")
print(f"  reconstruction_error: {final_metrics['reconstruction_error']:.4f}")

## 11. Fit final model & save artifacts

Calls the same `run_final_pipeline()` used by the CLI script.

In [ ]:
final_output_dir = OUTPUT_DIR / f"final_k{FINAL_K}"

result = run_final_pipeline(
    df=df,
    k=FINAL_K,
    output_dir=final_output_dir,
    text_columns=TEXT_COLUMNS,
    n_words=N_WORDS,
    naming_words=NAMING_WORDS,
    year_col=find_year_column(df),
    random_state=RANDOM_STATE,
    clean=CLEAN_TEXT,
)

result["keywords_df"]

## 12. Topic trends over time (if year column present)

In [ ]:
if result["trend"] is not None:
    shares = result["trend"]["shares"]
    shares.plot(kind="area", stacked=True, figsize=(14, 5), colormap="tab20")
    plt.title(f"Topic share of publications over time (k={FINAL_K})")
    plt.ylabel("Share of publications")
    plt.xlabel(result["trend"]["year_col"])
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.savefig(final_output_dir / "topic_trends.png", bbox_inches="tight")
    plt.show()
else:
    print("No year column detected — skipping trend plot.")

## 13. Production run (optional)

Once satisfied with `FINAL_K`, run the CLI for the same artifacts outside the notebook:

```bash
cd backend
python scripts/modeling/run_nmf_topic_modeling.py \
  --data data/processed/common/common_publications_final_with_linearsvm.csv \
  --output-dir data/processed/common/nmf \
  --k 20

# Or let the script sweep and pick best k by coherence:
python scripts/modeling/run_nmf_topic_modeling.py \
  --data data/processed/common/common_publications_final_with_linearsvm.csv \
  --output-dir data/processed/common/nmf
```